# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')  # To suppress display warnings from pandas

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata
# Print an overview from metadata
print(f"Dataset Name: {getattr(metadata, 'name', '')}\nDescription: {getattr(metadata, 'description', '')}\nIdentifier: {getattr(metadata, 'identifier', '')}")

## 2. Data Overview
Review available record sets, their fields and columns using the Croissant schema. Here all references use the `@id` field as per the Croissant specification.

In [ ]:
# List available record sets and the fields/columns they contain

def show_record_sets(ds):
    print("Record Sets (by @id):\n------------------------")
    record_set_ids = []
    for rs in ds.record_sets():
        print(f"@id: {rs.id}")
        print(f"  name: {rs.name}")
        field_ids = [f.id for f in getattr(rs, 'fields', [])]
        column_ids = [c.id for c in getattr(rs, 'columns', [])]
        print(f"  Fields: {field_ids}")
        print(f"  Columns: {column_ids}")
        record_set_ids.append(rs.id)
        print()
    return record_set_ids

record_set_ids = show_record_sets(dataset)
if record_set_ids:
    print(f"Total record sets: {len(record_set_ids)}")
else:
    print("No record sets found. Ensure the dataset contains tabular data.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

**Note:** In this dataset, record set `@id`s and field `@id`s are used for access.

**Example:** Use identified record set `@id` (e.g., `'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'`)

Below we extract data from all available record sets and display their columns.

In [ ]:
# Get all record set @id's from the overview

# If no record sets found in previous cell, re-run this to get the list
if not record_set_ids:
    record_set_ids = [rs.id for rs in dataset.record_sets()]

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set {record_set_id} with {len(df)} rows and columns:\n{df.columns.tolist()}\n")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

# Select one record set to preview its data
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]  # Select the first one available
    print(f"Preview of first 5 rows for {selected_record_set_id}:")
    display(dataframes[selected_record_set_id].head())
else:
    print("No record set dataframes were loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping using the chosen record set and numeric fields.

We will:
- Select a numeric field by its `@id` (for example `'cr:field/molecular_weight_Da'` corresponding to molecular weight in Daltons, if present)
- Filter records based on a threshold
- Normalize the selected numeric field
- Optionally group by a categorical field (such as protein accession number `@id` 'cr:field/accession' if available)

**Note:** Ensure to reference all field/column names by their real column name as loaded, which are the field `@id`s by Croissant convention.

In [ ]:
# Use one DataFrame for EDA
df = dataframes.get(selected_record_set_id)
if df is None or df.empty:
    print('No data available for EDA.')
else:
    # Find a likely numeric field for analysis
    possible_numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64'] or pd.api.types.is_numeric_dtype(df[col])]
    if not possible_numeric_fields:
        print("No numeric fields found for EDA.")
    else:
        numeric_field_id = possible_numeric_fields[0]  # e.g., 'cr:field/molecular_weight_Da'
        print(f"Using numeric field '{numeric_field_id}' for numeric analysis.")
        
        threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where '{numeric_field_id}' > mean ({threshold:.2f}): {len(filtered_df)} rows")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"First 5 normalized values for '{numeric_field_id}':")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Attempt to group by a likely categorical field (e.g., accession number or description)
        possible_group_fields = [col for col in df.columns if col != numeric_field_id and df[col].dtype == 'object']
        group_field = possible_group_fields[0] if possible_group_fields else None

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped and averaged '{numeric_field_id}' by '{group_field}':")
            display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and possible_numeric_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    
    # If grouping field exists, plot mean by group
    if group_field:
        top_groups = grouped_df.sort_values(numeric_field_id, ascending=False).head(10)
        plt.figure(figsize=(10, 5))
        sns.barplot(x=group_field, y=numeric_field_id, data=top_groups, palette="viridis")
        plt.title(f"Top 10 Mean {numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion
In this notebook, we explored the FAIR<sup>2</sup> Croissant-structured dataset “Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells”.
- We loaded dataset metadata and reviewed available record sets and fields using their Croissant `@id` values.
- We extracted tabular data from a selected record set, filtered and normalized a numeric field, and visualized its distribution.
- The notebook can be extended to deeper protein-level or sample-wise analyses using the provided record set and field IDs.

**Note**: For the latest schema and data, always consult the official [Croissant dataset URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).